# 저장 g25 신규 RGB269쌍 추출·내용 감사 — CPU
기존102쌍 RGB는 다시 다운로드하지 않습니다. 정확한 신규269 key(538PNG)와 감사한 raw pose269·62관절 candidate·g25 snapshot을 고정 입력으로 사용합니다.
새 CPU runtime에서 root의 고정 source/input ZIP 두 개만 준비합니다. 가용 RAM2GiB/disk4GiB가 필요하며 입력 ZIP/해제는128MiB,source는64MiB 이하입니다.
명시 indexed_stereo_pairs269_v1: body896MiB/4096HTTP/1800초/read2GiB/반환decode768MiB/RSS1GiB/work896MiB+회수ZIP768MiB/cache4MiB/batch32. 설치·업로드·인증과 최종 회수120초는 별도입니다.
538PNG payload647429967bytes/17묶음입니다. Range 계획654MiB는 실측이나 완료 보장이 아닙니다. 반환 decode cap은 선택 payload bytes이며 내부 seek 재해제 전체량을 계측하지 않습니다. Read/time/RSS cap을 함께 유지합니다.
기존 index를 auto_build=False로 복원해 물리 순서32개씩 한 번 추출하며 scan/index 수정/자동 반복은 금지합니다. 모든 PNG SHA/native1024 RGB decode·키대응을 통과해야269쌍 완료입니다.
대표 GT2D overlay는 사람 검토용입니다. 정답 누락 mask와 화면 밖/visibility를 구분하고 train_ready=false/validation0/timestamp·원motion 미검증을 유지합니다.
실패하면 원본 요청·인증·실행 셀을 다시 호출하지 말고 독립 회수 셀만 사용합니다. 완료/partial chunk·pose·보고서·overlay를 보존합니다. 실제 RGB 원격 실행과371진단 학습은 아직 하지 않았습니다.


In [ ]:
# Root가 검토·커밋한 source와 ready notebook revision/SHA를 채웁니다.
SOURCE_ZIP_SHA256 = ""
SOURCE_REVISION = ""
NOTEBOOK_REVISION = ""
INPUT_ZIP_SHA256 = "e4a30c47f240f333cff0e69f97d6db5e2a5903154eed806dcd3f5b02a2ea624d"
CONFIG_SHA256 = "c24400c1ffe38c282a45095f22642f9c327ab39e2ae57dc6538737ff688b04fc"
JOB_ID = "rgb269_pairs_001"
SOURCE_ROOT = "/content/rgb269_pairs_001-source"
INPUT_ROOT = "/content/rgb269_pairs_001-inputs"
WORKSPACE = "/content/rgb269_pairs_001-work"
RESULT_ZIP = "/content/rgb269_pairs_001_result.zip"
RECOVERY_ZIP = "/content/rgb269_pairs_001_preparation_result.zip"


In [ ]:
import hashlib, io, json, re, stat, subprocess, sys, zipfile
from pathlib import Path, PurePosixPath
from google.colab import files
assert re.fullmatch("[a-f0-9]{64}", SOURCE_ZIP_SHA256)
assert re.fullmatch("[a-f0-9]{64}", INPUT_ZIP_SHA256)
assert re.fullmatch("[a-f0-9]{64}", CONFIG_SHA256)
assert re.fullmatch("[a-f0-9]{40}", SOURCE_REVISION)
assert not Path(WORKSPACE).exists() and not Path(WORKSPACE).is_symlink(), "소비된 job: 준비/scan 재실행 금지"
import psutil, shutil
assert psutil.virtual_memory().available >= 2*1024**3, "가용 RAM2GiB 필요"
assert shutil.disk_usage(Path(SOURCE_ROOT).parent).free >= 4*1024**3, "가용 disk4GiB 필요"
print("1/2: 고정 source ZIP을 선택하세요.")
source_uploaded = files.upload()
print("2/2: 고정 input ZIP을 선택하세요.")
input_uploaded = files.upload()
def selected(uploaded, pin, label):
    rows = [b for b in uploaded.values() if hashlib.sha256(b).hexdigest() == pin]
    assert len(rows) == 1, label + " ZIP 누락 또는 고정 SHA 불일치"
    return rows[0]
def inspect_zip(raw, maximum):
    assert len(raw) <= maximum, "압축 ZIP 크기 상한"
    payloads, directories, names = {}, set(), set()
    with zipfile.ZipFile(io.BytesIO(raw)) as z:
        rows = z.infolist()
        assert len(rows) <= 2048 and sum(i.file_size for i in rows) <= maximum, "ZIP 해제 상한"
        for item in rows:
            name = item.filename.rstrip("/")
            path = PurePosixPath(name)
            assert name and name == path.as_posix() and not path.is_absolute()
            assert ".." not in path.parts and "\\" not in name and all(":" not in part for part in path.parts), "ZIP 경로 오류"
            assert name.casefold() not in names, "ZIP 중복 경로"
            names.add(name.casefold())
            kind = stat.S_IFMT(item.external_attr >> 16)
            assert kind in (0, stat.S_IFDIR if item.is_dir() else stat.S_IFREG), "ZIP 링크/특수파일 금지"
            for parent in path.parents:
                if str(parent) != ".": directories.add(parent.as_posix())
            data = z.read(item)  # 모든 member CRC를 추출 전에 검사
            assert len(data) == item.file_size
            if item.is_dir(): directories.add(name)
            else: payloads[name] = data
    assert not set(payloads) & directories, "ZIP 파일/폴더 충돌"
    return payloads, directories
# 두 ZIP 모두 검증하기 전에는 source/input/준비 디렉터리를 만들지 않는다.
source_data = inspect_zip(selected(source_uploaded, SOURCE_ZIP_SHA256, "source"), 64*1024**2)
input_data = inspect_zip(selected(input_uploaded, INPUT_ZIP_SHA256, "input"), 128*1024**2)
assert source_data[0]["SOURCE_REVISION.txt"].decode("utf-8").strip() == SOURCE_REVISION
input_config = json.loads(input_data[0]["config.json"])
config_raw = json.dumps(input_config, ensure_ascii=False, sort_keys=True, separators=(",", ":"), allow_nan=False).encode()
assert hashlib.sha256(config_raw).hexdigest() == CONFIG_SHA256
def existing_matches(destination, contents):
    target = Path(destination).absolute()
    for part in (target, *target.parents):
        assert not part.is_symlink() and not getattr(part, "is_junction", lambda:False)(), "추출 경로 링크 금지"
    if not target.exists(): return False
    assert target.is_dir(), "추출 대상이 디렉터리가 아님"
    files_seen, dirs_seen = set(), set()
    for count, path in enumerate(target.rglob("*")):
        assert count < 4096 and not path.is_symlink() and not getattr(path, "is_junction", lambda:False)()
        relative = path.relative_to(target).as_posix()
        if path.is_dir(): dirs_seen.add(relative)
        else:
            assert path.is_file() and relative in contents[0], "기존 추출본의 다른 파일"
            expected = contents[0][relative]
            assert path.stat().st_size == len(expected) and path.read_bytes() == expected, "기존 추출본 바이트 불일치"
            files_seen.add(relative)
    assert files_seen == set(contents[0]) and dirs_seen == contents[1], "기존 추출본 파일/폴더 집합 불일치"
    return True
source_path, input_path = Path(SOURCE_ROOT).absolute(), Path(INPUT_ROOT).absolute()
assert not source_path.is_relative_to(input_path) and not input_path.is_relative_to(source_path)
source_exists = existing_matches(SOURCE_ROOT, source_data)
input_exists = existing_matches(INPUT_ROOT, input_data)
def materialize(destination, contents, exists):
    if exists: return  # 정확히 같은 기존 추출본만 읽기 재사용
    target = Path(destination)
    target.mkdir(parents=True, exist_ok=False)
    for name in sorted(contents[1], key=lambda s:len(PurePosixPath(s).parts)):
        (target/name).mkdir(exist_ok=True)
    for name, data in contents[0].items():
        with (target/name).open("xb") as stream: stream.write(data)
materialize(SOURCE_ROOT, source_data, source_exists)
materialize(INPUT_ROOT, input_data, input_exists)
sys.dont_write_bytecode = True
sys.path.insert(0, SOURCE_ROOT)
subprocess.check_call([sys.executable, "-m", "pip", "install", "indexed_gzip==1.10.3", "psutil"])
import indexed_gzip
assert indexed_gzip.__version__ == "1.10.3"
del source_uploaded, input_uploaded, source_data, input_data


In [ ]:
# 인증 전에 이전 모듈·원본/입력 pin·269 key·예산을 검사합니다.
import sys
from pathlib import Path
for name,module in tuple(sys.modules.items()):
    if name.split(".")[0] in ("rewind_archive", "rewind_archive_index", "rewind_prepare", "rewind_discovery"):
        loaded = getattr(module, "__file__", None)
        assert loaded and Path(loaded).resolve().is_relative_to(Path(SOURCE_ROOT).resolve()), "이전 코드가 로드됨: 새 CPU 런타임에서 시작하세요"
sys.path.insert(0,SOURCE_ROOT)
from rewind_discovery.pairs import preflight
CONFIG, PIN = preflight(INPUT_ROOT)
assert PIN == CONFIG_SHA256 and CONFIG["job_id"] == JOB_ID and len(CONFIG["pairs"]) == 269
print({"job_id":JOB_ID,"pairs":len(CONFIG["pairs"]),"limits":CONFIG["limits"],"train_ready":False})


In [ ]:
# 한 번만 실행합니다. 실패·중단 후에는 회수 셀만 실행합니다.
from pathlib import Path
assert not Path(WORKSPACE).exists(), "소비된 job: 다시 실행하지 마세요"
from google.colab import auth
auth.authenticate_user()
import google.auth
import requests
from google.auth.transport.requests import AuthorizedSession, Request
from rewind_archive.request_policy import validate_no_hidden_retries
from urllib3.util.retry import Retry
credentials, _ = google.auth.default(scopes=["https://www.googleapis.com/auth/drive.readonly"])
def session_factory():
    auth_http = requests.Session()
    try:
        auth_http.trust_env = False
        for adapter in auth_http.adapters.values():
            adapter.max_retries = Retry(total=0, connect=0, read=0, redirect=0, status=0)
        session = AuthorizedSession(credentials, max_refresh_attempts=0, auth_request=Request(session=auth_http))
    except BaseException:
        auth_http.close()
        raise
    close_outer = session.close
    def close_all():
        try:
            close_outer()
        finally:
            auth_http.close()
    session.close = close_all
    try:
        session.trust_env = False
        for adapter in session.adapters.values():
            adapter.max_retries = Retry(total=0, connect=0, read=0, redirect=0, status=0)
        # 초기 metadata 호출 전 내부 auth transport까지 검사한다.
        validate_no_hidden_retries(session)
    except BaseException:
        session.close()
        raise
    return session
from rewind_discovery.pairs import run
REPORT = run(INPUT_ROOT, WORKSPACE, session_factory)
print({k:REPORT.get(k) for k in ("status","stop_reason","complete_pairs","target_pairs","body_bytes","requests","elapsed_seconds","index_unchanged")})


In [ ]:
# 인증·추출·원본 요청 없이 현재 partial과 완료 파일을 회수합니다.
import hashlib, json, sys, zipfile
from pathlib import Path
from google.colab import files
def file_sha(path):
    h=hashlib.sha256()
    with Path(path).open("rb") as stream:
        while data:=stream.read(65536):h.update(data)
    return h.hexdigest()
if (Path(SOURCE_ROOT)/"SOURCE_REVISION.txt").is_file() and (Path(INPUT_ROOT)/"input_manifest.json").is_file():
    assert (Path(SOURCE_ROOT)/"SOURCE_REVISION.txt").read_text().strip() == SOURCE_REVISION
    sys.path.insert(0,SOURCE_ROOT)
    from rewind_discovery.pairs import recover
    target=Path(RESULT_ZIP)
    if not target.exists():
        result=recover(INPUT_ROOT,WORKSPACE,target,expected_config_sha256=CONFIG_SHA256)
    with zipfile.ZipFile(target) as z:
        assert z.testzip() is None
        info=json.loads(z.read("recovery.json"))
        assert info["config_sha256"] == CONFIG_SHA256 and info["job_id"] == JOB_ID
else:
    # source/input 준비가 끝나지 않은 경우에도 REPORT 변수 없이 실패 상태를 회수합니다.
    target=Path(RECOVERY_ZIP)
    if not target.exists():
        with zipfile.ZipFile(target,"x",zipfile.ZIP_STORED) as z:
            z.writestr("recovery.json",json.dumps({"job_id":JOB_ID,"config_sha256":CONFIG_SHA256,
                "source_revision":SOURCE_REVISION,"notebook_revision":NOTEBOOK_REVISION,
                "status":"preparation_failed_execution_state_unknown","train_ready":False}))
    with zipfile.ZipFile(target) as z:
        assert z.testzip() is None
        info=json.loads(z.read("recovery.json"))
        assert info["config_sha256"] == CONFIG_SHA256 and info["job_id"] == JOB_ID
print("회수 ZIP",str(target),target.stat().st_size,file_sha(target))
files.download(str(target))
